In [5]:
import os
import pandas as pd
df = pd.read_csv('cookies_05052026_name_cleaned.csv')
df.head()

C:\Users\boett\AppData\Local\Temp\ipykernel_48588\390407667.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('cookies_05052026_name_cleaned.csv')


,name,name_cleaned,first_party_cookie,c
0,.thumbcache_fa5607ee107e2a8e0c319b41a88868c3,thumbcache,True,221
1,33acrossIdFp,33acrossIdFp,True,529
2,AAAproxySessionSecure,AAAproxySessionSecure,True,314
3,ADRUM_BTa,ADRUM_BTa,True,6950
4,AFCN,AFCN,True,199


In [15]:
import pandas as pd

def aggregate_cleaned_cookies(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fasst gleiche Cookies zusammen, wenn name_cleaned und first_party_cookie gleich sind,
    und addiert dabei die Spalte c.
    """

    return (
        df
        .groupby(["name_cleaned", "first_party_cookie"], as_index=False)
        .agg(c=("c", "sum"))
    )
df_aggregated = aggregate_cleaned_cookies(df)
df_aggregated = df_aggregated.sort_values(
    by="c",
    ascending=False
).reset_index(drop=True)
df_aggregated.head()

,name_cleaned,first_party_cookie,c
0,_ga,True,3998925
1,ga,True,854367
2,_ga,False,842255
3,tld,True,473908
4,AWSALB,True,388188


In [16]:
def select_entries(
    df: pd.DataFrame,
    condition_col: str = "condition",
    n_first_true: int = 50,
    n_random_true: int = 25,
    n_random_false: int = 25,
    random_state: int | None = None,
) -> pd.DataFrame:
    """
    Nimmt:
    1. die ersten 50 Einträge mit condition == True
    2. danach zufällig 25 weitere Einträge mit condition == True,
       die nicht unter den ersten 50 sind
    3. danach zufällig 25 Einträge mit condition == False,
       die noch nicht ausgewählt wurden
    """

    # 1. Erste 50 mit condition == True
    first_true = df[df[condition_col] == True].head(n_first_true)

    # Bereits ausgewählte Indizes merken
    selected_indices = set(first_true.index)

    # 2. Random 25 mit condition == True, aber nicht in den ersten 50
    remaining_true = df[
        (df[condition_col] == True) &
        (~df.index.isin(selected_indices))
    ]

    random_true = remaining_true.sample(
        n=n_random_true,
        random_state=random_state
    )

    selected_indices.update(random_true.index)

    # 3. Random 25 mit condition == False, nicht in bisherigen 75
    remaining_false = df[
        (df[condition_col] == False) &
        (~df.index.isin(selected_indices))
    ]

    random_false = remaining_false.sample(
        n=n_random_false,
        random_state=random_state
    )

    # Alles zusammenführen
    result = pd.concat([first_true, random_true, random_false])

    return result

selected_df = select_entries(
    df_aggregated,
    condition_col="first_party_cookie",
    random_state=21
)
selected_df.head()

,name_cleaned,first_party_cookie,c
0,_ga,True,3998925
1,ga,True,854367
3,tld,True,473908
4,AWSALB,True,388188
5,AWSALBCORS,True,352883


In [17]:
selected_df

,name_cleaned,first_party_cookie,c
0,_ga,True,3998925
1,ga,True,854367
3,tld,True,473908
4,AWSALB,True,388188
5,AWSALBCORS,True,352883
...,...,...,...
65764,rdc1747188859543,False,2
98289,rdc1747752187133,False,2
91496,rdc1747588266881,False,2
73967,rdc1747319740640,False,2
